# Clinical RAG — Ingestion (Colab)

Runs the same pipeline as `ingestion/run_ingestion.py`, but embeds on a Colab GPU
instead of CPU, then upserts straight to Supabase.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

**Secrets:** click the key icon in the left sidebar and add:
- `SUPABASE_URL`
- `SUPABASE_SERVICE_ROLE_KEY`
- `HF_TOKEN` (only needed if the dataset/config requires auth; PubMedQA is public so this can be blank)

Toggle notebook access on for each secret you add.

In [ ]:
!pip install -q transformers supabase datasets tqdm

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_SERVICE_ROLE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

HF_DATASET = 'qiaojin/PubMedQA'
HF_CONFIG = 'pqa_labeled'   # switch to 'pqa_artificial' for the full ~211k-row corpus
QUERY_MODEL = 'ncbi/MedCPT-Query-Encoder'
ARTICLE_MODEL = 'ncbi/MedCPT-Article-Encoder'
MAX_LENGTH = 512
EMBED_BATCH_SIZE = 64
UPSERT_BATCH_SIZE = 64

assert SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY, 'Add SUPABASE_URL / SUPABASE_SERVICE_ROLE_KEY as Colab secrets first.'

In [ ]:
# --- load dataset and flatten into passages (mirrors ingestion/load_dataset.py) ---
from datasets import load_dataset as hf_load_dataset

dataset = hf_load_dataset(HF_DATASET, HF_CONFIG, split='train', token=HF_TOKEN)
print(f'{len(dataset)} rows loaded.')

passages = []
for row in dataset:
    pubid = str(row['pubid'])
    question = row['question']
    contexts = row['context']['contexts']
    final_decision = row.get('final_decision', '')
    meshes = row.get('meshes', []) or []
    for ctx in contexts:
        if ctx and ctx.strip():
            passages.append({
                'pubid': pubid,
                'question': question,
                'content': ctx.strip(),
                'metadata': {'final_decision': final_decision, 'meshes': meshes},
            })
print(f'{len(passages)} passages before cleaning.')

In [ ]:
# --- clean + dedupe (mirrors ingestion/chunk.py) ---
import re, unicodedata

def _clean(text):
    text = unicodedata.normalize('NFKC', text)
    return re.sub(r'\s+', ' ', text).strip()

seen = set()
cleaned = []
for p in passages:
    text = _clean(p['content'])
    if len(text) < 20:
        continue
    key = text[:200]
    if key in seen:
        continue
    seen.add(key)
    cleaned.append({**p, 'content': text})

passages = cleaned
print(f'{len(passages)} passages after cleaning/dedup.')

In [ ]:
# --- load MedCPT article encoder onto GPU (mirrors embedding-service/model.py) ---
from transformers import AutoTokenizer, AutoModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
article_tokenizer = AutoTokenizer.from_pretrained(ARTICLE_MODEL)
article_model = AutoModel.from_pretrained(ARTICLE_MODEL).to(device)
article_model.eval()
print('Model loaded on', device)

In [ ]:
# --- embed all passages in batches ---
from tqdm.auto import tqdm

def embed_batch(texts):
    with torch.no_grad():
        encoded = article_tokenizer(
            texts, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt',
        ).to(device)
        outputs = article_model(**encoded)
        embeddings = outputs.last_hidden_state[:, 0, :]  # CLS token, per MedCPT convention
    return embeddings.cpu().tolist()

embeddings = []
for i in tqdm(range(0, len(passages), EMBED_BATCH_SIZE), desc='Embedding'):
    batch_texts = [p['content'] for p in passages[i:i + EMBED_BATCH_SIZE]]
    embeddings.extend(embed_batch(batch_texts))

print(f'{len(embeddings)} embeddings computed, dim={len(embeddings[0])}')

In [ ]:
# --- upsert to Supabase (mirrors ingestion/upsert.py) ---
from supabase import create_client

client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

for i in tqdm(range(0, len(passages), UPSERT_BATCH_SIZE), desc='Upserting'):
    batch_p = passages[i:i + UPSERT_BATCH_SIZE]
    batch_e = embeddings[i:i + UPSERT_BATCH_SIZE]
    rows = [
        {
            'pubid': p['pubid'],
            'question': p['question'],
            'content': p['content'],
            'metadata': p['metadata'],
            'embedding': emb,
        }
        for p, emb in zip(batch_p, batch_e)
    ]
    client.table('documents').insert(rows).execute()

print('Done.')

In [ ]:
# --- sanity check: row count + one similarity search ---
count_res = client.table('documents').select('id', count='exact').limit(1).execute()
print('Rows in documents table:', count_res.count)

sample = client.rpc('match_documents', {
    'query_embedding': embeddings[0],
    'match_count': 3,
}).execute()
for r in sample.data:
    print(r['pubid'], round(r['similarity'], 3), r['content'][:80])